# Apache Iceberg no AWS Glue — Notebook de Referência

Cobre o fluxo completo: criação de tabela, insert, update, delete, merge, schema evolution, time travel, rollback e manutenção.

**Pré-requisitos**
- Glue 4.0 com `--datalake-formats iceberg`
- Parâmetro `--conf spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions`
- Role com permissões em S3 e Glue Catalog

## 1. Configuração — Sessão Glue

In [ ]:
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2
%idle_timeout 60

In [ ]:
import sys
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from datetime import date

sc          = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark       = glueContext.spark_session

# ── Iceberg: ativa extensões e aponta para o Glue Catalog ──────────────
spark.conf.set(
    "spark.sql.extensions",
    "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
)
spark.conf.set("spark.sql.catalog.glue_catalog",
               "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set("spark.sql.catalog.glue_catalog.warehouse",
               "s3://meu-datalake/warehouse/")
spark.conf.set("spark.sql.catalog.glue_catalog.catalog-impl",
               "org.apache.iceberg.aws.glue.GlueCatalog")
spark.conf.set("spark.sql.catalog.glue_catalog.io-impl",
               "org.apache.iceberg.aws.s3.S3FileIO")

# catalog default — sem precisar prefixar em cada query
spark.sql("USE glue_catalog.workshop")
print("✓ Sessão Iceberg pronta")

## 2. DDL — Criar Tabela Iceberg

In [ ]:
# Cria a tabela de pedidos — particionada por mês da data do pedido
spark.sql("""
    CREATE TABLE IF NOT EXISTS pedidos (
        pedido_id      BIGINT       COMMENT 'PK do pedido',
        cliente_id     BIGINT,
        produto        STRING,
        status         STRING,
        valor          DECIMAL(18,2),
        dt_pedido      DATE         COMMENT 'data do pedido'
    )
    USING iceberg
    PARTITIONED BY (months(dt_pedido))
    LOCATION 's3://meu-datalake/warehouse/workshop/pedidos'
    TBLPROPERTIES (
        'format-version'  = '2',
        'write.parquet.compression-codec' = 'snappy',
        'write.metadata.metrics.default'  = 'full'
    )
""")

print("✓ Tabela criada")

## 3. Insert — Carga Inicial

In [ ]:
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import LongType, StringType, DecimalType, DateType
from decimal import Decimal

dados = [
    (1001, 42, "Notebook",  "APROVADO",  Decimal("4599.90"), date(2024,1,10)),
    (1002, 17, "Monitor",   "APROVADO",  Decimal("1299.00"), date(2024,1,15)),
    (1003, 42, "Teclado",   "CANCELADO", Decimal(" 299.90"), date(2024,2,3)),
    (1004, 88, "SSD 1TB",   "APROVADO",  Decimal(" 450.00"), date(2024,2,20)),
    (1005, 33, "Headset",   "PENDENTE",  Decimal(" 189.90"), date(2024,3,5)),
]

schema = StructType([
    StructField("pedido_id",  LongType()),
    StructField("cliente_id", LongType()),
    StructField("produto",    StringType()),
    StructField("status",     StringType()),
    StructField("valor",      DecimalType(18,2)),
    StructField("dt_pedido",  DateType()),
])

df = spark.createDataFrame(dados, schema)

# append — preserva dados existentes, Iceberg cria novo snapshot
df.writeTo("pedidos").append()

print(f"✓ {df.count()} linhas inseridas")

In [ ]:
spark.sql("SELECT * FROM pedidos ORDER BY pedido_id").show(truncate=False)

## 4. Update e Delete — Row-Level Operations

In [ ]:
# UPDATE — atualiza o status do pedido 1005 para APROVADO
spark.sql("""
    UPDATE pedidos
    SET status = 'APROVADO', valor = 209.90
    WHERE pedido_id = 1005
""")

# DELETE — remove pedidos cancelados
spark.sql("""
    DELETE FROM pedidos
    WHERE status = 'CANCELADO'
""")

spark.sql("SELECT pedido_id, status, valor FROM pedidos ORDER BY pedido_id").show()

## 5. Merge Into — Upsert

In [ ]:
# Novos pedidos chegando do staging
novos = [
    (1002, 17, "Monitor", "ENTREGUE", Decimal("1299.00"), date(2024,1,15)),
    (1006, 55, "Webcam",  "APROVADO", Decimal(" 349.90"), date(2024,3,22)),
]
df_novos = spark.createDataFrame(novos, schema)
df_novos.createOrReplaceTempView("staging")

spark.sql("""
    MERGE INTO pedidos t
    USING staging s
        ON t.pedido_id = s.pedido_id
    WHEN MATCHED THEN
        UPDATE SET t.status = s.status
    WHEN NOT MATCHED THEN
        INSERT *
""")

spark.sql("SELECT * FROM pedidos ORDER BY pedido_id").show(truncate=False)

## 6. Schema Evolution — Alter Table

In [ ]:
# Adiciona coluna nova — sem reescrever dados existentes
spark.sql("ALTER TABLE pedidos ADD COLUMN desconto DECIMAL(10,2)")

# Renomeia coluna — suportado no Iceberg, quebraria no Hive
spark.sql("ALTER TABLE pedidos RENAME COLUMN produto TO nome_produto")

# Confere o schema atualizado
spark.sql("DESCRIBE TABLE pedidos").show(truncate=False)

## 7. Time Travel — Snapshots

In [ ]:
# Lista todos os snapshots gerados pelas operações anteriores
spark.sql("""
    SELECT
        snapshot_id,
        committed_at,
        operation,
        summary['added-records']   AS added,
        summary['deleted-records'] AS deleted
    FROM pedidos.snapshots
    ORDER BY committed_at
""").show(truncate=False)

In [ ]:
# Lê o estado da tabela ANTES do UPDATE/DELETE — snapshot original
# Substitua o snapshot_id pelo valor real retornado na célula anterior
SNAPSHOT_ORIGINAL = "3821047291034"  # <- ajuste

df_passado = (
    spark.read
    .option("snapshot-id", SNAPSHOT_ORIGINAL)
    .table("pedidos")
)
print(f"Linhas no snapshot original: {df_passado.count()}")
df_passado.select("pedido_id", "status").show()

# Alternativa por timestamp
df_ts = (
    spark.read
    .option("as-of-timestamp", "2024-03-22 09:02:00")
    .table("pedidos")
)
print(f"Linhas via timestamp: {df_ts.count()}")

In [ ]:
# Rollback para o snapshot original (desfaz UPDATE + DELETE)
# Substitua pelo snapshot_id real
spark.sql("""
    CALL glue_catalog.system.rollback_to_snapshot(
        'workshop.pedidos',
        3821047291034
    )
""")
print("✓ Rollback executado — tabela voltou ao estado inicial")
spark.sql("SELECT COUNT(*) AS total FROM pedidos").show()

## 8. Manutenção — Compactação e Limpeza

In [ ]:
# Compacta arquivos pequenos gerados pelos writes incrementais
# target: 128 MB por arquivo de saída
resultado = spark.sql("""
    CALL glue_catalog.system.rewrite_data_files(
        table    => 'workshop.pedidos',
        strategy => 'binpack',
        options  => map(
            'target-file-size-bytes', '134217728',
            'min-file-size-bytes',    '67108864'
        )
    )
""")
resultado.show()

In [ ]:
# Expira snapshots antigos — mantém os últimos 5 e os de até 7 dias atrás
from datetime import datetime, timedelta, timezone

cutoff = datetime.now(timezone.utc) - timedelta(days=7)

spark.sql(f"""
    CALL glue_catalog.system.expire_snapshots(
        table       => 'workshop.pedidos',
        older_than  => TIMESTAMP '{cutoff.strftime('%Y-%m-%d %H:%M:%S')}',
        retain_last => 5
    )
""")

# Remove arquivos órfãos no S3 (sem referência nos manifests)
spark.sql("""
    CALL glue_catalog.system.remove_orphan_files(
        table => 'workshop.pedidos'
    )
""")

print("✓ Manutenção concluída")

## 9. Inspeção — Arquivos e Partições

In [ ]:
# Inspeciona os arquivos físicos por partição (útil para diagnóstico)
spark.sql("""
    SELECT
        partition,
        file_count,
        file_size_in_bytes,
        record_count
    FROM pedidos.partitions
    ORDER BY partition
""").show(truncate=False)

In [ ]:
# Propriedades e metadados da tabela
spark.sql("SHOW TBLPROPERTIES pedidos").show(truncate=False)